# PPO + SMC Asset Allocation — Colab GPU 訓練

本 notebook 將在 Colab 上使用 GPU 訓練 PPO 策略，作為論文 Findings 章節的數據來源。

**前置條件（重要）**：
1. 在 Colab 選單：`Runtime → Change runtime type → Hardware accelerator: T4 GPU`（免費版）或更高。
2. repo 必須是 public（已確認）或你有 GitHub Personal Access Token。
3. 訓練資料（7 個 Parquet 快照，552 KB）已 commit 在 repo 內，clone 即帶下來，**不需要**另外上傳。

**預估時間**：100k step × T4 GPU ≈ 5–10 分鐘。

## Step 1 — Clone repo 並切到目標 branch

In [ ]:
# repo 的 003-ppo-training-env branch 含 PortfolioEnv（spec 003）+ PPO trainer（spec 004 MVP）。
# 若你已 merge 到 main，可改 -b main。
!git clone -b 003-ppo-training-env --depth 1 https://github.com/DaYi-TW/ppo-smc-asset-allocation.git
%cd ppo-smc-asset-allocation
!git log --oneline -5

## Step 2 — 安裝相依（含 GPU 版 PyTorch）

Colab 的 base image 已內建 PyTorch + CUDA，所以這裡只需要 `pip install -e .` 把 repo 自身註冊為 editable package，再加裝 stable-baselines3。

In [ ]:
!pip install -q -e .
!pip install -q stable-baselines3 tensorboard

In [ ]:
# 驗證 GPU 可用
import torch
print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 3 — 驗證資料快照（憲法 Principle I 可重現性 gate）

本步驟比對 `data/raw/` 下 7 個 Parquet 的 SHA-256 與 metadata sidecar；不符會 fail。

In [ ]:
!python -m data_ingestion.cli verify

## Step 4 — 啟動 TensorBoard（背景）

在訓練前啟動，訓練過程中即可看到 reward 三項分量、loss、entropy 等曲線即時更新。

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/

## Step 5 — 訓練 PPO（含 SMC 特徵）

**參數說明**：
- `--total-timesteps 100000`：100k step。完整論文實驗建議 500k–1M。
- `--seed 42`：固定 seed，可重現。多 seed 訓練可改變這個值。
- `--device cuda`：強制使用 GPU。如果 CUDA 不可用會 raise（不靜默退回 CPU）。
- `--metrics-freq 1000`：每 1000 step 寫一列 `metrics.csv`。

輸出位置：`runs/<UTC_timestamp>_<git_hash>_seed<N>/`。

In [ ]:
!python -m ppo_training.train \
    --total-timesteps 100000 \
    --seed 42 \
    --device cuda \
    --metrics-freq 1000

## Step 6 — （可選）SMC ablation 對照訓練

為了量化 SMC 特徵的增量價值，跑一次純價格 baseline（observation 從 63 維 → 33 維）。其他超參數完全一致。

**注意**：這會把訓練時間翻倍。如果 GPU 有時間限制（Colab free tier 12 hr/session），先跑完 SMC 版再決定是否跑這個。

In [ ]:
!python -m ppo_training.train \
    --total-timesteps 100000 \
    --seed 42 \
    --device cuda \
    --no-smc \
    --metrics-freq 1000

## Step 7 — 檢查 artefacts

In [ ]:
!ls -la runs/
print("\n--- 最後一個 run 的 metadata ---")
import json, glob, os
latest = sorted(glob.glob('runs/*'))[-1]
print(f"latest: {latest}\n")
with open(os.path.join(latest, 'metadata.json')) as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

In [ ]:
# 預覽 metrics.csv
import pandas as pd
import glob
latest = sorted(glob.glob('runs/*'))[-1]
df = pd.read_csv(f'{latest}/metrics.csv')
print(f"metrics rows: {len(df)}")
df.tail(10)

## Step 7.5 — 評估訓練好的 policy（業務級指標）

把 `final_policy.zip` 載回 env 跑完整 episode，計算 NAV / Sharpe / MDD / 年化報酬。
這是論文 Findings 章節最關鍵的數字 — `final_mean_episode_return` 是 PPO 訓練 reward 的指標，不是業務報酬。

預設 `deterministic=True`（同 policy + 同資料 → 完全相同軌跡，憲法 Principle I）。

In [ ]:
import glob
latest = sorted(glob.glob('runs/*'))[-1]
!python -m ppo_training.evaluate \
    --policy {latest}/final_policy.zip \
    --data-root data/raw \
    --save-trajectory

In [ ]:
# 視覺化 NAV 曲線 + 部位配置
import pandas as pd
import matplotlib.pyplot as plt
import glob

latest = sorted(glob.glob('runs/*'))[-1]
traj = pd.read_csv(f'{latest}/trajectory.csv', parse_dates=['date'])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax1.plot(traj['date'], traj['nav'], lw=1.5, color='steelblue', label='PPO policy NAV')
ax1.axhline(1.0, ls='--', color='gray', alpha=0.5, label='Initial NAV')
peaks = traj['nav'].cummax()
ax1.fill_between(traj['date'], traj['nav'], peaks, where=(traj['nav'] < peaks),
                 color='red', alpha=0.15, label='Drawdown')
ax1.set_ylabel('NAV')
ax1.set_title(f'PPO + SMC policy — full-episode NAV ({latest.split("/")[-1]})')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

weight_cols = [c for c in traj.columns if c.startswith('w_')]
ax2.stackplot(traj['date'], [traj[c] for c in weight_cols],
              labels=[c.replace('w_', '') for c in weight_cols], alpha=0.85)
ax2.set_ylabel('Allocation weight')
ax2.set_xlabel('Date')
ax2.legend(loc='upper left', ncol=4, fontsize=8)
ax2.set_ylim(0, 1)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{latest}/nav_and_weights.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'圖已存到 {latest}/nav_and_weights.png')

## Step 8 — 打包並下載到本機

Colab session 結束後 runtime 會被回收，所以一定要把 `runs/` 下載回本機。

In [ ]:
!zip -r runs.zip runs/
!ls -lh runs.zip

In [ ]:
from google.colab import files
files.download('runs.zip')

## 後續步驟

下載回本機後，把 `runs/` 解壓到 repo 根目錄。後續可以：

1. **論文圖**：用 `metrics.csv` 畫訓練曲線（matplotlib / seaborn）。
2. **回測**：用 `final_policy.zip` 載入 policy，跑同 env 的另一段時間區間做 out-of-sample backtest。
3. **多 seed 實驗**：改 `--seed 0,1,2,3,4` 重複跑 5 次（spec 004 US2，論文必需）。
4. **比較 SMC vs no-SMC**：算 5 seed × 2 condition 的 mean ± std，做 Welch's t-test。

## 疑難排解

- `data_hashes mismatch`：你的 local repo 有人改過 `data/raw/`。重 clone 即可。
- `CUDA out of memory`：把 `--n-steps 2048 --batch-size 64` 調小（這個任務 obs/action 都很小，不應該爆——若爆代表 Colab 給你的 GPU 太弱）。
- `git clone` 失敗：repo 是否還是 public？切到 `--branch main` 試試。